In [0]:
# PySpark Functions
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    round,
    when,
    regexp_replace,
    to_date,
    to_timestamp,
    lit,
    current_timestamp
)

# Alias for round
from pyspark.sql.functions import round as spark_round

# PySpark Types
from pyspark.sql.types import *

# Python Libraries
import datetime
import logging
import builtins

In [0]:
#silver_energy_usage
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_energy_usage")

# ──────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────
STREAM_NAME   = "energy_usage"
SOURCE_TABLE  = "energy_catalog.raw.bronze_energy_usage"
BUCKET        = "s3://energy-pipline-prod"
SILVER_PATH   = f"{BUCKET}/silver/energy_usage/"
CATALOG_TABLE = "energy_catalog.processed.silver_energy_usage"
batch_date    = datetime.date.today().strftime("%Y-%m-%d")

try:
    logger.info(f"Starting Silver transformation — {STREAM_NAME}")
    logger.info(f"Batch date : {batch_date}")
    logger.info(f"Source     : {SOURCE_TABLE}")
    logger.info(f"Target     : {CATALOG_TABLE}")

    # ──────────────────────────────────────────────
    # Check source has data
    # ──────────────────────────────────────────────
    try:
        source_count = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {SOURCE_TABLE}"
        ).collect()[0]["cnt"]
    except Exception:
        logger.warning(f"Source table not found: {SOURCE_TABLE}")
        source_count = 0

    if source_count == 0:
        logger.warning("No records in source. Silver transformation skipped.")

    else:
        logger.info(f"Records detected : {source_count}")

        # ──────────────────────────────────────────
        # Read from Bronze
        # ──────────────────────────────────────────
        df = spark.table(SOURCE_TABLE)
        logger.info("Bronze data loaded successfully")

        # ──────────────────────────────────────────
        # Step 1 — Drop audit columns BEFORE dedup
        # _ingestion_ts makes every row unique
        # so audit cols must be dropped first
        # ──────────────────────────────────────────
        df = df.drop("_batch_date", "_source_file", "_stream_name", "_ingestion_ts")
        logger.info("Audit columns dropped")

        # ──────────────────────────────────────────
        # Step 2 — Full row dedup
        # energy_usage has only 5 true duplicates
        # ──────────────────────────────────────────
        count_before = df.count()
        df = df.dropDuplicates()
        count_after = df.count()
        logger.info(f"Dedup complete — removed {count_before - count_after} duplicate rows")
        logger.info(f"Rows after dedup : {count_after}")   # expect ~49,995

        # ──────────────────────────────────────────
        # Step 3 — Drop rows with nulls
        # 1,000 null rows across 10 numeric columns
        # ~2% data loss — acceptable
        # ──────────────────────────────────────────
        count_before_null_drop = df.count()

        df = df.dropna(subset=[
            "voltage_reading",
            "current_reading",
            "active_power_kw",
            "reactive_power_kvar",
            "energy_usage_kwh",
            "frequency_hz",
            "load_factor",
            "peak_demand_kw",
            "offpeak_demand_kw",
            "daily_consumption_kwh"
        ])

        count_after_null_drop = df.count()
        dropped_nulls = count_before_null_drop - count_after_null_drop
        logger.info(f"Null rows dropped          : {dropped_nulls}")             # expect ~1,000
        logger.info(f"Rows after null drop       : {count_after_null_drop}")     # expect ~48,995

        null_check = df.filter(col("energy_usage_kwh").isNull()).count()
        if null_check > 0:
            logger.warning(f"Nulls still present after drop : {null_check}")
        else:
            logger.info("NULL check passed — 0 nulls remaining")

        # ──────────────────────────────────────────
        # Step 4 — Parse timestamp string → TimestampType
        # Actual format in data: dd-MM-yyyy HH:mm
        # Example value: 30-08-2023 16:40
        # ──────────────────────────────────────────
        df = df.withColumn("event_timestamp",
            to_timestamp(col("timestamp"), "dd-MM-yyyy HH:mm"))

        null_ts = df.filter(col("event_timestamp").isNull()).count()
        if null_ts > 0:
            logger.warning(f"Unparseable timestamps : {null_ts} rows")
        else:
            logger.info("Timestamp parsing complete — 0 parse failures")

        df = df.drop("timestamp")
        logger.info("Raw timestamp string column dropped")

        # ──────────────────────────────────────────
        # Step 5 — Trim and standardize text
        # ──────────────────────────────────────────
        for c in ["region_name", "city_name", "meter_type",
                  "customer_category", "grid_zone"]:
            df = df.withColumn(c, trim(col(c)))

        df = df.withColumn("region_name",       upper(col("region_name"))) \
               .withColumn("city_name",         upper(col("city_name"))) \
               .withColumn("customer_category", upper(col("customer_category")))
        logger.info("Text columns trimmed and standardized to UPPER")

        # ──────────────────────────────────────────
        # Step 6 — Remove invalid records
        # energy_usage_kwh must be > 0
        # ──────────────────────────────────────────
        invalid_count = df.filter(col("energy_usage_kwh") <= 0).count()
        df = df.filter(col("energy_usage_kwh") > 0)
        logger.info(f"Invalid records removed (energy_usage_kwh <= 0) : {invalid_count}")

        # ──────────────────────────────────────────
        # Step 7 — Derived metrics
        # power_factor = active_power_kw / reactive_power_kvar
        # ──────────────────────────────────────────
        df = df.withColumn("power_factor",
            round(
                col("active_power_kw") / when(col("reactive_power_kvar") == 0, 0.001)
                                          .otherwise(col("reactive_power_kvar")),
                4
            )
        )
        logger.info("Derived metric added : power_factor")

        df = df.withColumn("is_outlier",
            when(col("energy_usage_kwh") > 10000, True).otherwise(False))
        logger.info("Outlier flag added : is_outlier")

        # ──────────────────────────────────────────
        # Step 8 — Ensure Silver schema exists
        # ──────────────────────────────────────────
        spark.sql("CREATE SCHEMA IF NOT EXISTS energy_catalog.processed")
        logger.info("Schema energy_catalog.processed verified")

        # ──────────────────────────────────────────
        # Step 9 — Write Silver Delta
        # Overwrite — Silver is always a clean rebuilt view
        # ──────────────────────────────────────────
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .partitionBy("region_name") \
            .save(SILVER_PATH)
        logger.info(f"Delta write complete → {SILVER_PATH}")

        # ──────────────────────────────────────────
        # Step 10 — Register in Unity Catalog
        # ──────────────────────────────────────────
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
            USING DELTA
            LOCATION '{SILVER_PATH}'
        """)
        logger.info(f"Table registered : {CATALOG_TABLE}")

        # ──────────────────────────────────────────
        # Step 11 — Verify
        # ──────────────────────────────────────────
        written = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}"
        ).collect()[0]["cnt"]
        logger.info(f"Rows in Silver table : {written}")

        null_kwh = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}
            WHERE energy_usage_kwh IS NULL
        """).collect()[0]["cnt"]

        null_pf = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}
            WHERE power_factor IS NULL
        """).collect()[0]["cnt"]

        null_ts_final = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}
            WHERE event_timestamp IS NULL
        """).collect()[0]["cnt"]

        if null_kwh > 0:
            logger.warning(f"Nulls in energy_usage_kwh : {null_kwh}")
        else:
            logger.info("NULL check passed — energy_usage_kwh : 0 nulls")

        if null_pf > 0:
            logger.warning(f"Nulls in power_factor : {null_pf}")
        else:
            logger.info("NULL check passed — power_factor : 0 nulls")

        if null_ts_final > 0:
            logger.warning(f"Nulls in event_timestamp : {null_ts_final}")
        else:
            logger.info("NULL check passed — event_timestamp : 0 nulls")

        logger.info(f"Silver {STREAM_NAME} — SUCCESSFUL ✓")

except Exception as e:
    logger.error(f"Silver transformation FAILED — {STREAM_NAME}")
    logger.error(str(e))
    raise

finally:
    logger.info(f"Silver pipeline finished — {STREAM_NAME}")

In [0]:
#Silver_device_metrics
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_device_metrics")

# ──────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────
STREAM_NAME   = "device_metrics"
SOURCE_TABLE  = "energy_catalog.raw.bronze_device_metrics"
BUCKET        = "s3://energy-pipline-prod"
SILVER_PATH   = f"{BUCKET}/silver/device_metrics/"
CATALOG_TABLE = "energy_catalog.processed.silver_device_metrics"
batch_date    = datetime.date.today().strftime("%Y-%m-%d")

try:
    logger.info(f"Starting Silver transformation — {STREAM_NAME}")
    logger.info(f"Batch date : {batch_date}")
    logger.info(f"Source     : {SOURCE_TABLE}")
    logger.info(f"Target     : {CATALOG_TABLE}")

    # ──────────────────────────────────────────────
    # Check source has data
    # ──────────────────────────────────────────────
    try:
        source_count = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {SOURCE_TABLE}"
        ).collect()[0]["cnt"]
    except Exception:
        logger.warning(f"Source table not found: {SOURCE_TABLE}")
        source_count = 0

    if source_count == 0:
        logger.warning("No records in source. Silver transformation skipped.")

    else:
        logger.info(f"Records detected : {source_count}")

        # ──────────────────────────────────────────
        # Read from Bronze
        # ──────────────────────────────────────────
        df = spark.table(SOURCE_TABLE)
        logger.info("Bronze data loaded successfully")

        # ──────────────────────────────────────────
        # Step 1 — Drop audit columns first
        # ──────────────────────────────────────────
        df = df.drop("_batch_date", "_source_file", "_stream_name", "_ingestion_ts")
        logger.info("Audit columns dropped")

        # ──────────────────────────────────────────
        # Step 2 — Business key dedup with rounded runtime
        # Round runtime_hours to avoid float precision false positives
        # Removes 176 duplicates
        # ──────────────────────────────────────────
        count_before = df.count()

        df = df.withColumn("runtime_hours_rounded", spark_round("runtime_hours", 0))
        df = df.dropDuplicates([
            "household_id", "device_category", "device_brand",
            "device_model", "runtime_hours_rounded"
        ]).drop("runtime_hours_rounded")

        count_after = df.count()
        logger.info(f"Dedup complete — removed {count_before - count_after} duplicate rows")
        logger.info(f"Rows after dedup : {count_after}")   # expect ~49,824

        # ──────────────────────────────────────────
        # Step 3 — No nulls in device_metrics
        # Confirm and log — if unexpected nulls found, drop them
        # ──────────────────────────────────────────
        null_check = df.filter(col("efficiency_ratio").isNull()).count()
        if null_check > 0:
            logger.warning(f"Unexpected nulls in efficiency_ratio : {null_check}")
            df = df.dropna(subset=["efficiency_ratio", "runtime_hours",
                                   "energy_draw_kwh", "device_power_kw"])
            logger.info("Unexpected nulls dropped")
        else:
            logger.info("NULL check passed — 0 nulls in device_metrics")

        # ──────────────────────────────────────────
        # Step 4 — Trim and standardize text
        # ──────────────────────────────────────────
        for c in ["device_category", "device_brand", "device_model",
                  "maintenance_status", "installation_region"]:
            df = df.withColumn(c, trim(upper(col(c))))
        logger.info("Text columns trimmed and standardized to UPPER")

        # ──────────────────────────────────────────
        # Step 5 — Clamp efficiency_ratio to 0.5–1.0
        # ──────────────────────────────────────────
        df = df.withColumn("efficiency_ratio",
            when(col("efficiency_ratio") < 0.5, 0.5)
            .when(col("efficiency_ratio") > 1.0, 1.0)
            .otherwise(col("efficiency_ratio")))
        logger.info("efficiency_ratio clamped to valid range 0.5–1.0")

        # ──────────────────────────────────────────
        # Step 6 — Derived metric
        # efficiency_score = energy_draw_kwh / runtime_hours
        # ──────────────────────────────────────────
        df = df.withColumn("efficiency_score",
            round(
                col("energy_draw_kwh") / when(col("runtime_hours") == 0, 0.001)
                                          .otherwise(col("runtime_hours")),
                4
            )
        )
        logger.info("Derived metric added : efficiency_score")

        # ──────────────────────────────────────────
        # Step 7 — Ensure Silver schema exists
        # ──────────────────────────────────────────
        spark.sql("CREATE SCHEMA IF NOT EXISTS energy_catalog.processed")
        logger.info("Schema energy_catalog.processed verified")

        # ──────────────────────────────────────────
        # Step 8 — Write Silver Delta
        # ──────────────────────────────────────────
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .partitionBy("installation_region") \
            .save(SILVER_PATH)
        logger.info(f"Delta write complete → {SILVER_PATH}")

        # ──────────────────────────────────────────
        # Step 9 — Register in Unity Catalog
        # ──────────────────────────────────────────
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
            USING DELTA
            LOCATION '{SILVER_PATH}'
        """)
        logger.info(f"Table registered : {CATALOG_TABLE}")

        # ──────────────────────────────────────────
        # Step 10 — Verify
        # ──────────────────────────────────────────
        written = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}"
        ).collect()[0]["cnt"]
        logger.info(f"Rows in Silver table : {written}")

        null_es = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}
            WHERE efficiency_score IS NULL
        """).collect()[0]["cnt"]

        if null_es > 0:
            logger.warning(f"Nulls in efficiency_score : {null_es}")
        else:
            logger.info("NULL check passed — efficiency_score : 0 nulls")

        logger.info(f"Silver {STREAM_NAME} — SUCCESSFUL ✓")

except Exception as e:
    logger.error(f"Silver transformation FAILED — {STREAM_NAME}")
    logger.error(str(e))
    raise

finally:
    logger.info(f"Silver pipeline finished — {STREAM_NAME}")

In [0]:
#silver_grid_load
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_grid_load")

# ──────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────
STREAM_NAME   = "grid_load"
SOURCE_TABLE  = "energy_catalog.raw.bronze_grid_load"
BUCKET        = "s3://energy-pipline-prod"
SILVER_PATH   = f"{BUCKET}/silver/grid_load/"
CATALOG_TABLE = "energy_catalog.processed.silver_grid_load"
batch_date    = datetime.date.today().strftime("%Y-%m-%d")

try:
    logger.info(f"Starting Silver transformation — {STREAM_NAME}")
    logger.info(f"Batch date : {batch_date}")
    logger.info(f"Source     : {SOURCE_TABLE}")
    logger.info(f"Target     : {CATALOG_TABLE}")

    # ──────────────────────────────────────────────
    # Check source has data
    # ──────────────────────────────────────────────
    try:
        source_count = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {SOURCE_TABLE}"
        ).collect()[0]["cnt"]
    except Exception:
        logger.warning(f"Source table not found: {SOURCE_TABLE}")
        source_count = 0

    if source_count == 0:
        logger.warning("No records in source. Silver transformation skipped.")

    else:
        logger.info(f"Records detected : {source_count}")

        # ──────────────────────────────────────────
        # Read from Bronze
        # ──────────────────────────────────────────
        df = spark.table(SOURCE_TABLE)
        logger.info("Bronze data loaded successfully")

        # ──────────────────────────────────────────
        # Step 1 — Drop audit columns first
        # ──────────────────────────────────────────
        df = df.drop("_batch_date", "_source_file", "_stream_name", "_ingestion_ts")
        logger.info("Audit columns dropped")

        # ──────────────────────────────────────────
        # Step 2 — Infrastructure identity key dedup
        # ──────────────────────────────────────────
        count_before = df.count()

        df = df.dropDuplicates([
            "household_id",
            "grid_region",
            "substation_name",
            "feeder_line",
            "distribution_zone",
            "grid_operator"
        ])

        count_after = df.count()
        logger.info(f"Dedup complete — removed {count_before - count_after} duplicate rows")
        logger.info(f"Rows after dedup : {count_after}")

        # ──────────────────────────────────────────
        # Step 3 — No nulls in grid_load
        # Confirm and log — if unexpected nulls found, drop them
        # ──────────────────────────────────────────
        null_check = df.filter(col("grid_load_kw").isNull()).count()
        if null_check > 0:
            logger.warning(f"Unexpected nulls in grid_load_kw : {null_check}")
            df = df.dropna(subset=["grid_load_kw", "transformer_load",
                                   "grid_capacity_kw", "reserve_margin"])
            logger.info("Unexpected nulls dropped")
        else:
            logger.info("NULL check passed — 0 nulls in grid_load")

        # ──────────────────────────────────────────
        # Step 4 — Trim and standardize text
        # ──────────────────────────────────────────
        for c in ["grid_region", "substation_name", "feeder_line",
                  "distribution_zone", "grid_operator"]:
            df = df.withColumn(c, trim(upper(col(c))))
        logger.info("Text columns trimmed and standardized to UPPER")

        # ──────────────────────────────────────────
        # Step 5 — Confirm negative values preserved
        # load_variation (-20 to +20) and
        # frequency_variation (-0.5 to +0.5)
        # are legitimate grid measurements — NOT errors
        # ──────────────────────────────────────────
        neg_load = df.filter(col("load_variation") < 0).count()
        neg_freq = df.filter(col("frequency_variation") < 0).count()
        logger.info(f"Negative load_variation rows (expected)      : {neg_load}")
        logger.info(f"Negative frequency_variation rows (expected) : {neg_freq}")

        # ──────────────────────────────────────────
        # Step 6 — Derived metrics
        # ──────────────────────────────────────────
        df = df.withColumn("reserve_margin_flag",
            when(col("reserve_margin") < 5,  "CRITICAL")
            .when(col("reserve_margin") < 10, "WARNING")
            .otherwise("NORMAL"))
        logger.info("Derived metric added : reserve_margin_flag")

        df = df.withColumn("grid_stress_ratio",
            round(
                col("grid_load_kw") / when(col("transformer_load") == 0, 0.001)
                                       .otherwise(col("transformer_load")),
                4
            )
        )
        logger.info("Derived metric added : grid_stress_ratio")

        # ──────────────────────────────────────────
        # Step 7 — Ensure Silver schema exists
        # ──────────────────────────────────────────
        spark.sql("CREATE SCHEMA IF NOT EXISTS energy_catalog.processed")
        logger.info("Schema energy_catalog.processed verified")

        # ──────────────────────────────────────────
        # Step 8 — Write Silver Delta
        # ──────────────────────────────────────────
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .partitionBy("grid_region") \
            .save(SILVER_PATH)
        logger.info(f"Delta write complete → {SILVER_PATH}")

        # ──────────────────────────────────────────
        # Step 9 — Register in Unity Catalog
        # ──────────────────────────────────────────
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
            USING DELTA
            LOCATION '{SILVER_PATH}'
        """)
        logger.info(f"Table registered : {CATALOG_TABLE}")

        # ──────────────────────────────────────────
        # Step 10 — Verify
        # ──────────────────────────────────────────
        written = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}"
        ).collect()[0]["cnt"]
        logger.info(f"Rows in Silver table : {written}")

        critical_count = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}
            WHERE reserve_margin_flag = 'CRITICAL'
        """).collect()[0]["cnt"]

        if critical_count > 0:
            logger.warning(f"CRITICAL reserve margin rows : {critical_count}")
        else:
            logger.info("Reserve margin check — 0 CRITICAL rows")

        logger.info(f"Silver {STREAM_NAME} — SUCCESSFUL ✓")

except Exception as e:
    logger.error(f"Silver transformation FAILED — {STREAM_NAME}")
    logger.error(str(e))
    raise

finally:
    logger.info(f"Silver pipeline finished — {STREAM_NAME}")

In [0]:
#silver_tariff_metrics
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_tariff_metrics")

# ──────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────
STREAM_NAME   = "tariff_metrics"
SOURCE_TABLE  = "energy_catalog.raw.bronze_tariff_metrics"
BUCKET        = "s3://energy-pipline-prod"
SILVER_PATH   = f"{BUCKET}/silver/tariff_metrics/"
CATALOG_TABLE = "energy_catalog.processed.silver_tariff_metrics"
batch_date    = datetime.date.today().strftime("%Y-%m-%d")

try:
    logger.info(f"Starting Silver transformation — {STREAM_NAME}")
    logger.info(f"Batch date : {batch_date}")
    logger.info(f"Source     : {SOURCE_TABLE}")
    logger.info(f"Target     : {CATALOG_TABLE}")

    # ──────────────────────────────────────────────
    # Check source has data
    # ──────────────────────────────────────────────
    try:
        source_count = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {SOURCE_TABLE}"
        ).collect()[0]["cnt"]
    except Exception:
        logger.warning(f"Source table not found: {SOURCE_TABLE}")
        source_count = 0

    if source_count == 0:
        logger.warning("No records in source. Silver transformation skipped.")

    else:
        logger.info(f"Records detected : {source_count}")

        # ──────────────────────────────────────────
        # Read from Bronze
        # ──────────────────────────────────────────
        df = spark.table(SOURCE_TABLE)
        logger.info("Bronze data loaded successfully")

        # ──────────────────────────────────────────
        # Step 1 — Drop audit columns first
        # ──────────────────────────────────────────
        df = df.drop("_batch_date", "_source_file", "_stream_name", "_ingestion_ts")
        logger.info("Audit columns dropped")

        # ──────────────────────────────────────────
        # Step 2 — Business key dedup
        # Removes 951 duplicates
        # ──────────────────────────────────────────
        count_before = df.count()

        df = df.dropDuplicates([
            "household_id", "tariff_region", "tariff_city",
            "tariff_plan_type", "utility_provider"
        ])

        count_after = df.count()
        logger.info(f"Dedup complete — removed {count_before - count_after} duplicate rows")
        logger.info(f"Rows after dedup : {count_after}")   # expect ~49,049

        # ──────────────────────────────────────────
        # Step 3 — No nulls in tariff_metrics
        # Confirm and log — if unexpected nulls found, drop them
        # ──────────────────────────────────────────
        null_check = df.filter(col("monthly_bill").isNull()).count()
        if null_check > 0:
            logger.warning(f"Unexpected nulls in monthly_bill : {null_check}")
            df = df.dropna(subset=["monthly_bill", "billing_units",
                                   "unit_rate", "adjustment_amount"])
            logger.info("Unexpected nulls dropped")
        else:
            logger.info("NULL check passed — 0 nulls in tariff_metrics")

        # ──────────────────────────────────────────
        # Step 4 — Trim and standardize text
        # ──────────────────────────────────────────
        for c in ["tariff_region", "tariff_city", "tariff_plan_type",
                  "billing_cycle", "utility_provider"]:
            df = df.withColumn(c, trim(upper(col(c))))
        logger.info("Text columns trimmed and standardized to UPPER")

        # ──────────────────────────────────────────
        # Step 5 — Confirm negative adjustment_amount preserved
        # Negative values = billing credits — NOT errors
        # Do NOT drop or abs() these values
        # ──────────────────────────────────────────
        neg_adj = df.filter(col("adjustment_amount") < 0).count()
        logger.info(f"Negative adjustment_amount rows (billing credits, expected) : {neg_adj}")

        min_adj = df.agg({"adjustment_amount": "min"}).collect()[0][0]
        if min_adj > 0:
            logger.warning("No negative adjustment_amount found — billing credits may have been lost")
        else:
            logger.info(f"Billing credits preserved — min adjustment_amount : {builtins.round(min_adj, 2)}")

        # ──────────────────────────────────────────
        # Step 6 — Derived metric
        # effective_rate = monthly_bill / billing_units
        # ──────────────────────────────────────────
        df = df.withColumn("effective_rate",
            round(
                col("monthly_bill") / when(col("billing_units") == 0, 0.001)
                                       .otherwise(col("billing_units")),
                4
            )
        )
        logger.info("Derived metric added : effective_rate")

        # ──────────────────────────────────────────
        # Step 7 — Ensure Silver schema exists
        # ──────────────────────────────────────────
        spark.sql("CREATE SCHEMA IF NOT EXISTS energy_catalog.processed")
        logger.info("Schema energy_catalog.processed verified")

        # ──────────────────────────────────────────
        # Step 8 — Write Silver Delta
        # ──────────────────────────────────────────
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .partitionBy("tariff_region") \
            .save(SILVER_PATH)
        logger.info(f"Delta write complete → {SILVER_PATH}")

        # ──────────────────────────────────────────
        # Step 9 — Register in Unity Catalog
        # ──────────────────────────────────────────
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
            USING DELTA
            LOCATION '{SILVER_PATH}'
        """)
        logger.info(f"Table registered : {CATALOG_TABLE}")

        # ──────────────────────────────────────────
        # Step 10 — Verify
        # ──────────────────────────────────────────
        written = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}"
        ).collect()[0]["cnt"]
        logger.info(f"Rows in Silver table : {written}")

        null_er = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}
            WHERE effective_rate IS NULL
        """).collect()[0]["cnt"]

        if null_er > 0:
            logger.warning(f"Nulls in effective_rate : {null_er}")
        else:
            logger.info("NULL check passed — effective_rate : 0 nulls")

        logger.info(f"Silver {STREAM_NAME} — SUCCESSFUL ✓")

except Exception as e:
    logger.error(f"Silver transformation FAILED — {STREAM_NAME}")
    logger.error(str(e))
    raise

finally:
    logger.info(f"Silver pipeline finished — {STREAM_NAME}")

In [0]:
#silver_weather
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_weather")

# ──────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────
STREAM_NAME   = "weather"
SOURCE_TABLE  = "energy_catalog.raw.bronze_weather"
BUCKET        = "s3://energy-pipline-prod"
SILVER_PATH   = f"{BUCKET}/silver/weather/"
CATALOG_TABLE = "energy_catalog.processed.silver_weather"
batch_date    = datetime.date.today().strftime("%Y-%m-%d")

try:
    logger.info(f"Starting Silver transformation — {STREAM_NAME}")
    logger.info(f"Batch date : {batch_date}")
    logger.info(f"Source     : {SOURCE_TABLE}")
    logger.info(f"Target     : {CATALOG_TABLE}")

    # ──────────────────────────────────────────────
    # Check source has data
    # ──────────────────────────────────────────────
    try:
        source_count = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {SOURCE_TABLE}"
        ).collect()[0]["cnt"]
    except Exception:
        logger.warning(f"Source table not found: {SOURCE_TABLE}")
        source_count = 0

    if source_count == 0:
        logger.warning("No records in source. Silver transformation skipped.")

    else:
        logger.info(f"Records detected : {source_count}")

        # ──────────────────────────────────────────
        # Read from Bronze
        # ──────────────────────────────────────────
        df = spark.table(SOURCE_TABLE)
        logger.info("Bronze data loaded successfully")

        # ──────────────────────────────────────────
        # Step 1 — Drop audit columns BEFORE dedup
        # ──────────────────────────────────────────
        df = df.drop("_batch_date", "_source_file", "_stream_name", "_ingestion_ts")
        logger.info("Audit columns dropped")

        # ──────────────────────────────────────────
        # Step 2 — Full row dedup
        # weather has 0 true duplicates
        # ──────────────────────────────────────────
        count_before = df.count()
        df = df.dropDuplicates()
        count_after = df.count()
        logger.info(f"Dedup complete — removed {count_before - count_after} duplicate rows")
        logger.info(f"Rows after dedup : {count_after}")   # expect 50,000

        # ──────────────────────────────────────────
        # Step 3 — Fix dirty condition_type
        # Data has: 'Cloudy%', 'Rainy%', 'Sunny%'
        # Strip trailing % character
        # ──────────────────────────────────────────
        dirty_count = df.filter(col("condition_type").contains("%")).count()
        logger.info(f"Dirty condition_type rows (with %) : {dirty_count}")

        df = df.withColumn("condition_type",
            regexp_replace(col("condition_type"), "%", ""))

        remaining_dirty = df.filter(col("condition_type").contains("%")).count()
        if remaining_dirty > 0:
            logger.warning(f"Dirty condition_type rows still present : {remaining_dirty}")
        else:
            logger.info("condition_type cleaned — % suffix removed successfully")

        # ──────────────────────────────────────────
        # Step 4 — Drop rows with nulls
        # 1,000 null rows across 10 numeric columns
        # ~2% data loss — acceptable
        # ──────────────────────────────────────────
        count_before_null_drop = df.count()

        df = df.dropna(subset=[
            "temperature_celsius",
            "humidity_percent",
            "wind_speed_kmh",
            "rainfall_mm",
            "pressure_hpa",
            "solar_radiation",
            "dew_point",
            "uv_index",
            "visibility_km",
            "cloud_cover_percent"
        ])

        count_after_null_drop = df.count()
        dropped_nulls = count_before_null_drop - count_after_null_drop
        logger.info(f"Null rows dropped          : {dropped_nulls}")             # expect ~1,000
        logger.info(f"Rows after null drop       : {count_after_null_drop}")     # expect ~49,000

        null_check = df.filter(col("temperature_celsius").isNull()).count()
        if null_check > 0:
            logger.warning(f"Nulls still present after drop : {null_check}")
        else:
            logger.info("NULL check passed — 0 nulls remaining")

        # ──────────────────────────────────────────
        # Step 5 — Parse timestamp
        # weather format: dd-MM-yyyy
        # Example: 01-01-2023
        # NO time component — date only
        # ──────────────────────────────────────────
        df = df.withColumn("weather_date",
            to_date(col("timestamp"), "dd-MM-yyyy"))

        null_date = df.filter(col("weather_date").isNull()).count()
        if null_date > 0:
            logger.warning(f"Unparseable dates : {null_date} rows")
        else:
            logger.info("Date parsing complete — 0 parse failures")

        df = df.drop("timestamp")
        logger.info("Raw timestamp string column dropped")

        # ──────────────────────────────────────────
        # Step 6 — Trim and standardize text
        # ──────────────────────────────────────────
        for c in ["weather_region", "weather_city", "weather_station",
                  "climate_zone", "condition_type"]:
            df = df.withColumn(c, trim(upper(col(c))))
        logger.info("Text columns trimmed and standardized to UPPER")

        # ──────────────────────────────────────────
        # Step 7 — Derived metric
        # weather_factor = temperature_celsius * humidity_percent
        # ──────────────────────────────────────────
        df = df.withColumn("weather_factor",
            round(col("temperature_celsius") * col("humidity_percent"), 2))
        logger.info("Derived metric added : weather_factor")

        # ──────────────────────────────────────────
        # Step 8 — Ensure Silver schema exists
        # ──────────────────────────────────────────
        spark.sql("CREATE SCHEMA IF NOT EXISTS energy_catalog.processed")
        logger.info("Schema energy_catalog.processed verified")

        # ──────────────────────────────────────────
        # Step 9 — Write Silver Delta
        # ──────────────────────────────────────────
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .partitionBy("weather_region") \
            .save(SILVER_PATH)
        logger.info(f"Delta write complete → {SILVER_PATH}")

        # ──────────────────────────────────────────
        # Step 10 — Register in Unity Catalog
        # ──────────────────────────────────────────
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
            USING DELTA
            LOCATION '{SILVER_PATH}'
        """)
        logger.info(f"Table registered : {CATALOG_TABLE}")

        # ──────────────────────────────────────────
        # Step 11 — Verify
        # ──────────────────────────────────────────
        written = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}"
        ).collect()[0]["cnt"]
        logger.info(f"Rows in Silver table : {written}")

        dirty_remaining = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {CATALOG_TABLE}
            WHERE condition_type LIKE '%\\%%'
        """).collect()[0]["cnt"]

        if dirty_remaining > 0:
            logger.warning(f"Dirty condition_type still present : {dirty_remaining} rows")
        else:
            logger.info("Dirty data check passed — no % in condition_type")

        distinct_conditions = spark.sql(f"""
            SELECT DISTINCT condition_type FROM {CATALOG_TABLE}
        """).collect()
        conditions = [r["condition_type"] for r in distinct_conditions]
        logger.info(f"Distinct condition_type values : {conditions}")
        # Expected: ['CLOUDY', 'RAINY', 'SUNNY']

        logger.info(f"Silver {STREAM_NAME} — SUCCESSFUL ✓")

except Exception as e:
    logger.error(f"Silver transformation FAILED — {STREAM_NAME}")
    logger.error(str(e))
    raise

finally:
    logger.info(f"Silver pipeline finished — {STREAM_NAME}")

In [0]:
%sql
-- Row counts
SELECT 'silver_energy_usage'   AS stream, COUNT(*) AS rows FROM energy_catalog.processed.silver_energy_usage   UNION ALL
SELECT 'silver_device_metrics' AS stream, COUNT(*) AS rows FROM energy_catalog.processed.silver_device_metrics UNION ALL
SELECT 'silver_grid_load'      AS stream, COUNT(*) AS rows FROM energy_catalog.processed.silver_grid_load      UNION ALL
SELECT 'silver_tariff_metrics' AS stream, COUNT(*) AS rows FROM energy_catalog.processed.silver_tariff_metrics UNION ALL
SELECT 'silver_weather'        AS stream, COUNT(*) AS rows FROM energy_catalog.processed.silver_weather;

In [0]:
%sql
-- Weather must show no % signs
SELECT DISTINCT condition_type FROM energy_catalog.processed.silver_weather;
-- Expected: CLOUDY, RAINY, SUNNY

In [0]:
%sql
-- Billing credits must be negative
SELECT MIN(adjustment_amount), MAX(adjustment_amount)
FROM energy_catalog.processed.silver_tariff_metrics;
-- Min must be negative (~-49.99)

In [0]:
%sql
-- Zero nulls in energy_usage
SELECT
    SUM(CASE WHEN energy_usage_kwh IS NULL THEN 1 ELSE 0 END) AS null_kwh,
    SUM(CASE WHEN power_factor     IS NULL THEN 1 ELSE 0 END) AS null_pf,
    SUM(CASE WHEN event_timestamp  IS NULL THEN 1 ELSE 0 END) AS null_ts
FROM energy_catalog.processed.silver_energy_usage;
-- All must be 0